In [26]:
import h5py
import networkx as nx
import numpy as np
from collections import defaultdict

file_path = "./SMA_data_processing/cobre_connectomes_database.h5"

graphs_hc = []
graphs_scz = []

with h5py.File(file_path, "r") as f:
    hc = f["hc_matrices"]
    scz = f["scz_matrices"]

    for i in range(hc.shape[0]):
        A = hc[i]
        A = A.copy()
        np.fill_diagonal(A, 0)
        graphs_hc.append(nx.from_numpy_array(A))

    for i in range(scz.shape[0]):
        A = scz[i]
        A = A.copy()
        np.fill_diagonal(A, 0)
        graphs_scz.append(nx.from_numpy_array(A))

print("HC graphs:", len(graphs_hc))
print("SCZ graphs:", len(graphs_scz))
print(graphs_hc[0])
print(list(graphs_hc[0].neighbors(0)))

G = graphs_hc[0]

HC graphs: 70
SCZ graphs: 68
Graph with 219 nodes and 1385 edges
[3, 5, 7, 9, 11, 13, 52, 70, 95, 101, 102, 103, 106, 108, 109, 113, 142, 176, 197]


Reference : https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.community.louvain.louvain_communities.html

The algorithm works in 2 steps. On the first step it assigns every node to be in its own community and then for each node it tries to find the maximum positive modularity gain by moving each node to all of its neighbor communities. If no positive gain is achieved the node remains in its original community.

The modularity gain obtained by moving an **isolated node** into a community can easily be calculated by the following formula :

$$
\Delta Q = \frac{k_{i,in}}{2m} - \gamma \frac{\Sigma_{tot}\cdot k_i}{2m^2}
$$

where :

- $m$ is the size of the graph
- $k_{i,\mathrm{in}}$ is the sum of the weights of the links from $i$ to nodes in $C$
- $k_i$ is the sum of the weights of the links incident to node $i$
- $\Sigma_{\mathrm{tot}}$ is the sum of the weights of the links incident to nodes in $C$
- $\gamma$ is the resolution parameter


In [27]:
def initial_assignment_communities(G) :
    return {node: node for node in G.nodes()}

In [28]:
def delta_q(G, node, target_community, communities, gamma=1.0, weight="weight"):

    degrees = dict(G.degree(weight=weight))
    two_m = sum(degrees.values())

    k_i_in = 0.0

    for neighbor, edge_data in G[node].items():
        if neighbor != node and communities[neighbor] == target_community:
            k_i_in += edge_data.get(weight)

    k_i = degrees[node]

    sigma_tot = 0.0

    for n in G.nodes():
        if n != node and communities[n] == target_community:
            sigma_tot += degrees[n]

    return (k_i_in / two_m) - gamma * (sigma_tot * k_i) / (two_m * two_m)

In [29]:
def louvain_step_one(G, gamma=1.0, weight="weight"):
    
    communities = initial_assignment_communities(G)
    moved = True

    while moved:
        moved = False

        for node in G.nodes():
            current_community = communities[node]

            # change community to isolate node
            communities[node] = -1
            
            #unique values
            target_communities = set()

            for neighbor in G.neighbors(node):
                if communities[neighbor] != -1:
                    target_communities.add(communities[neighbor])

            best_community = current_community
            best_dq = 0.0

            for tc in target_communities:
                dq = delta_q(G, node, tc, communities, gamma=gamma, weight=weight)
                if dq > best_dq:
                    best_dq = dq
                    best_community = tc

            communities[node] = best_community

            if best_community != current_community:
                moved = True

    return communities

In [30]:
communities = louvain_step_one(G)
print(communities)

{0: 108, 1: 105, 2: 104, 3: 108, 4: 107, 5: 197, 6: 105, 7: 135, 8: 107, 9: 108, 10: 105, 11: 108, 12: 108, 13: 105, 14: 152, 15: 152, 16: 152, 17: 121, 18: 152, 19: 121, 20: 25, 21: 125, 22: 143, 23: 25, 24: 143, 25: 25, 26: 143, 27: 25, 28: 125, 29: 125, 30: 105, 31: 114, 32: 114, 33: 143, 34: 143, 35: 167, 36: 114, 37: 114, 38: 142, 39: 143, 40: 25, 41: 68, 42: 68, 43: 68, 44: 152, 45: 164, 46: 152, 47: 86, 48: 152, 49: 86, 50: 184, 51: 152, 52: 156, 53: 125, 54: 58, 55: 190, 56: 58, 57: 58, 58: 58, 59: 152, 60: 182, 61: 167, 62: 167, 63: 167, 64: 171, 65: 167, 66: 174, 67: 171, 68: 167, 69: 68, 70: 108, 71: 177, 72: 178, 73: 184, 74: 182, 75: 184, 76: 78, 77: 78, 78: 78, 79: 142, 80: 182, 81: 182, 82: 86, 83: 190, 84: 86, 85: 184, 86: 86, 87: 190, 88: 184, 89: 179, 90: 194, 91: 180, 92: 182, 93: 182, 94: 180, 95: 197, 96: 198, 97: 177, 98: 198, 99: 101, 100: 105, 101: 101, 102: 105, 103: 108, 104: 104, 105: 105, 106: 197, 107: 107, 108: 108, 109: 197, 110: 105, 111: 105, 112: 108, 

The second phase consists in building a new network whose nodes are now the communities found in the first phase. To do so, the weights of the links between the new nodes are given by the sum of the weight of the links between nodes in the corresponding two communities. Once this phase is complete it is possible to reapply the first phase creating bigger communities with increased modularity.

In [31]:
def louvain_step_two(G, communities, weight="weight"):
    new_G = nx.Graph()

    unique_communities = set(communities.values())
    for uc in unique_communities:
        new_G.add_node(uc)

    for n1, n2, edge_data in G.edges(data=True):
        c_n1 = communities[n1]
        c_n2 = communities[n2]
        w = edge_data.get(weight)

        if new_G.has_edge(c_n1, c_n2):
            new_G[c_n1][c_n2][weight] += w
        else:
            new_G.add_edge(c_n1, c_n2, **{weight: w})

    return new_G

In [32]:
new_G = louvain_step_two(G, communities)
print("New : ", new_G)
print("Original :", G)

New :  Graph with 42 nodes and 288 edges
Original : Graph with 219 nodes and 1385 edges


The above two phases are executed until no modularity gain is achieved (or is less than the threshold, or until max_levels is reached).

$$
Q = \frac{1}{2m} \sum_{ij} \left( A_{ij} - \frac{k_i k_j}{2m} \right)\delta(c_i, c_j)
$$

In [33]:
def louvain(G, weight="weight", threshold=1e-04, max_levels=1000):
    

_IncompleteInputError: incomplete input (3906651039.py, line 2)